## Методы Монте-Карло и машинное обучение

 Домашнее задание: Обучить нейросетевую модель на базе регулируемого трансформера (выбрать любую задачу и любые данные) и применить оптимизацию гиперпараметров с использованием методов Монте-Карло

 Оценить и проанализировать результаты научных экспериментов

 Сделать выводы

Выполнил: Барышев Даниил, 3 курс

В своём дз хочу попробовать метод байесовской оптимизации через Optuna на двух датасетах

In [1]:
!pip install optuna datasets transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.3 MB/s eta 0:00:00


## Начнём с простого датасета Digits

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna
import numpy as np

digits = load_digits()
X, y = digits.data, digits.target
X = X.reshape(-1, 8, 8)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
test_ds = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))

class TinyTransformer(nn.Module):
    def __init__(self, input_dim, num_heads, num_layers, hidden_dim, num_classes):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=num_heads,
                                                   dim_feedforward=hidden_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(input_dim * 8, num_classes)

    def forward(self, x):
        x = self.transformer(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

def train_and_eval(model, train_loader, test_loader, lr, epochs=10):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            outputs = model(batch_x)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    return correct / total

default_model = TinyTransformer(input_dim=8, num_heads=2, num_layers=1, hidden_dim=32, num_classes=10)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)
accuracy_default_digits = train_and_eval(default_model, train_loader, test_loader, lr=0.001)

def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    hidden_dim = trial.suggest_int("hidden_dim", 16, 64)
    model = TinyTransformer(input_dim=8, num_heads=2, num_layers=num_layers,
                            hidden_dim=hidden_dim, num_classes=10)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=32)

    accuracy = train_and_eval(model, train_loader, test_loader, lr)
    return accuracy

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("\nРезультаты анализа:")
print(f"Точность по умолчанию: {accuracy_default_digits:.4f}")
print(f"Лучшая точность: {study.best_value:.4f}")
print(f"Лучшие гиперпараметры: {study.best_params}")

[I 2026-05-10 20:37:42,686] A new study created in memory with name: no-name-c94a29b4-d02a-43fe-9392-67d3ae8181f8
[I 2026-05-10 20:37:48,121] Trial 0 finished with value: 0.9666666666666667 and parameters: {'lr': 0.00508554732544784, 'num_layers': 3, 'hidden_dim': 32}. Best is trial 0 with value: 0.9666666666666667.
[I 2026-05-10 20:37:52,842] Trial 1 finished with value: 0.9666666666666667 and parameters: {'lr': 0.0011428335714517366, 'num_layers': 3, 'hidden_dim': 49}. Best is trial 0 with value: 0.9666666666666667.
[I 2026-05-10 20:37:58,900] Trial 2 finished with value: 0.9805555555555555 and parameters: {'lr': 0.0051350098779780985, 'num_layers': 2, 'hidden_dim': 23}. Best is trial 2 with value: 0.9805555555555555.
[I 2026-05-10 20:38:02,446] Trial 3 finished with value: 0.9 and parameters: {'lr': 0.00040678086587726315, 'num_layers': 2, 'hidden_dim': 39}. Best is trial 2 with value: 0.9805555555555555.
[I 2026-05-10 20:38:05,360] Trial 4 finished with value: 0.9444444444444444 an


Результаты анализа:
Точность по умолчанию: 0.9556
Лучшая точность: 0.9833
Лучшие гиперпараметры: {'lr': 0.004952499980056382, 'num_layers': 2, 'hidden_dim': 25}


## Попробуем датасет по-сложнее: эмоциональный окрас предложения

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

emotions = load_dataset('dair-ai/emotion', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=16)

emotions_encoded = emotions.map(tokenize, batched=True, batch_size=None)
X_hf = torch.tensor(emotions_encoded['train']['input_ids'][:2000])
y_hf = torch.tensor(emotions_encoded['train']['label'][:2000])

X_train_hf, X_test_hf, y_train_hf, y_test_hf = train_test_split(X_hf.numpy(), y_hf.numpy(), test_size=0.2, random_state=42)

hf_train_ds = TensorDataset(torch.LongTensor(X_train_hf), torch.LongTensor(y_train_hf))
hf_test_ds = TensorDataset(torch.LongTensor(X_test_hf), torch.LongTensor(y_test_hf))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse th

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
class TextTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
                                                   dim_feedforward=hidden_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

default_model = TextTransformer(vocab_size=tokenizer.vocab_size, embed_dim=32, num_heads=4, num_layers=2, hidden_dim=128, num_classes=6)
train_loader = DataLoader(hf_train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(hf_test_ds, batch_size=32)
accuracy_default = train_and_eval(default_model, train_loader, test_loader, lr=1e-3, epochs=5)

def objective_hf(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    embed_dim = trial.suggest_categorical("embed_dim", [32, 64])
    model = TextTransformer(vocab_size=tokenizer.vocab_size, embed_dim=embed_dim,
                            num_heads=4, num_layers=2, hidden_dim=128, num_classes=6)


    return train_and_eval(model, train_loader, test_loader, lr, epochs=5)

study_hf = optuna.create_study(direction="maximize")
study_hf.optimize(objective_hf, n_trials=10)

print(f"\nРезультаты для Hugging Face (Emotion):")
print(f"Точность по умолчанию: {accuracy_default:.4f}")
print(f"Лучшая точность: {study_hf.best_value:.4f}")
print(f"Параметры: {study_hf.best_params}")

[I 2026-05-10 20:39:30,346] A new study created in memory with name: no-name-fb9762ad-5134-41d6-91e0-5fca45c2cd96
[I 2026-05-10 20:39:39,450] Trial 0 finished with value: 0.3 and parameters: {'lr': 1.329368935096815e-05, 'embed_dim': 32}. Best is trial 0 with value: 0.3.
[I 2026-05-10 20:39:55,099] Trial 1 finished with value: 0.335 and parameters: {'lr': 0.00018588881785859497, 'embed_dim': 32}. Best is trial 1 with value: 0.335.
[I 2026-05-10 20:40:04,450] Trial 2 finished with value: 0.3725 and parameters: {'lr': 0.0008349738660165313, 'embed_dim': 32}. Best is trial 2 with value: 0.3725.
[I 2026-05-10 20:40:13,947] Trial 3 finished with value: 0.33 and parameters: {'lr': 4.329622525720372e-05, 'embed_dim': 32}. Best is trial 2 with value: 0.3725.
[I 2026-05-10 20:40:23,923] Trial 4 finished with value: 0.3525 and parameters: {'lr': 0.0005339381830667484, 'embed_dim': 32}. Best is trial 2 with value: 0.3725.
[I 2026-05-10 20:40:43,618] Trial 5 finished with value: 0.345 and paramete


Результаты для Hugging Face (Emotion):
Точность по умолчанию: 0.3850
Лучшая точность: 0.3925
Параметры: {'lr': 0.0003127819538629242, 'embed_dim': 64}


### Итоговый отчет по экспериментам

Посмотрел на работу простеньких нейронок с блоком трансформера для задач распознавания чисел и эмоциональности текста, использовал Optuna для подбора параметров


| Датасет | Точность (Default) | Точность (Optuna) | Улучшение |
| :--- | :---: | :---: | :---: |
| **Digits** | 0.9556 |  0.9833 | **+2.77%** |
| **Emotion** | 0.385 | 0.3925 | **+0.75%** |

#### Ключевые выводы
- Трансформеры крайне чувствительны к гиперпараметрам. Даже небольшое изменение `learning rate` и `hidden_dim` в датасете Digits позволило сократить ошибку на 3%
- В отличие от Grid Search, алгоритм не тратит ресурсы на заведомо плохие комбинации, а фокусируется на областях, дающих максимальный прирост точности.
- Из-за рандомной природы подбор гиперпараметров произошёл очень быстро

#### Заключение
Использование стохастических методов оптимизации гиперпараметров нужно и полезно для быстрой оценки работоспособности модели